# Retail Sales Forecasting & Business Intelligence — Milestone 2
**Participant:** Participant_M2_03  
**Project:** Retail Sales Forecasting & Business Intelligence  
**Milestone:** 2  
**Dataset package:** `Retail_(TS).zip`

## Objective
Forecast daily sales for each store-item combination and support inventory/replenishment decisions.

## Milestone scope
lag/rolling and calendar features; Decision Tree, Random Forest, and boosting regressors; time-aware tuning; compare MAE, RMSE, sMAPE; inspect residuals and feature importance.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## Time-aware ML comparison
Lag and rolling features must use only past observations. Split chronologically and report errors overall and by store/item.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv(Path("data") / "retail_data.csv", parse_dates=["date"])
df = df.sort_values(["store", "item", "date"])
g = df.groupby(["store", "item"])["sales"]
df["lag_1"] = g.shift(1); df["lag_7"] = g.shift(7)
df["rolling_7"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())
df["dow"] = df["date"].dt.dayofweek; df["month"] = df["date"].dt.month
data = df.dropna(subset=["lag_1", "lag_7", "rolling_7"])
cutoff = data["date"].sort_values().iloc[int(len(data)*.8)]
train, test = data[data.date <= cutoff], data[data.date > cutoff]
features = ["store", "item", "lag_1", "lag_7", "rolling_7", "dow", "month"]
rows = []
for name, model in {
    "DecisionTree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=150, max_depth=16, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42)
}.items():
    model.fit(train[features], train["sales"])
    pred = model.predict(test[features])
    rows.append({"model": name, "MAE": mean_absolute_error(test.sales, pred),
                 "RMSE": mean_squared_error(test.sales, pred)**.5})
display(pd.DataFrame(rows).sort_values("RMSE"))

## Limitations
This is a baseline comparison, not exhaustive hyperparameter tuning. Check the split strategy and report per-segment errors before selecting a model.